# 03 · FP16 Baselines for the Five Core Models

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Update 4 goal:** the accuracy of every core model in full precision (FP16). Every quantized run later is compared with these numbers.

| Model | Size | GPUs |
|---|---|---|
| `Qwen/Qwen3-1.7B` (thinking off) | 1.7B | one T4 |
| `Qwen/Qwen3-4B` (thinking off) | 4B | one T4 |
| `Qwen/Qwen3-8B` (thinking off) | 8B | both T4s (tensor parallel) |
| `meta-llama/Llama-3.2-1B-Instruct` | 1B | one T4 |
| `meta-llama/Llama-3.2-3B-Instruct` | 3B | one T4 |

Each model answers all 1,390 BFCL `non_live` questions in free mode, with our own script (`src/run_eval.py`, checked in Update 3), and is scored with BFCL's AST checker (`src/score.py`).

## How to run on Kaggle

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. **Required for Llama:** **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face *read* token from huggingface.co → Settings → Access Tokens) and switch it on for this notebook.
4. Click **Run All** to watch it, or **Save Version → Save & Run All (Commit)** to let it run in the background.

**Time:** about 1-1.5 hours in total. Models run one after another; if one fails, the others still run.
**Output:** `update4_results.zip` with the raw answers, scores, run information and `baseline_fp16.md`.

## 1. Settings
Edit this cell only.

In [ ]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run

# Run order: the four single-GPU models first, Qwen3-8B (both GPUs) last
CONFIGS = [
    "configs/qwen3-1.7b_fp16.yaml",
    "configs/qwen3-4b_fp16.yaml",
    "configs/llama-3.2-1b_fp16.yaml",
    "configs/llama-3.2-3b_fp16.yaml",
    "configs/qwen3-8b_fp16.yaml",
]
MODE = "free"
REPORT_NAME = "baseline_fp16.md"
REPORT_TITLE = "FP16 Baselines (Update 4)"
RUN_TIMEOUT_MIN = 90          # a run that takes longer than this is stopped, and the next model starts
DELETE_MODEL_AFTER_RUN = True # frees disk space: the five models take about 37 GB together

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # separate Python 3.11 environment (not kept)


def run_id_of(config):
    return os.path.splitext(os.path.basename(config))[0] + f"_{MODE}"

## 2. Check the GPU, internet and Hugging Face token

In [ ]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)
if len(smi.stdout.strip().splitlines()) < 2:
    print("WARNING: only one GPU. Qwen3-8B needs both T4s (Settings -> Accelerator -> GPU T4 x2).")

try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: NOT SET. Llama 3.2 cannot be downloaded without it (Add-ons -> Secrets).")

## 3. Get the code and check access to every model
Clones the repository, then downloads one small file of each model at the pinned revision. This finds a missing token or missing Llama approval now, not after an hour of running.

In [ ]:
import yaml
from huggingface_hub import hf_hub_download

shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

cfgs = {}
problems = []
for config in CONFIGS:
    with open(f"{REPO}/{config}") as f:
        cfgs[config] = yaml.safe_load(f)
    model, revision = cfgs[config]["model"], str(cfgs[config]["revision"])
    try:
        hf_hub_download(model, "config.json", revision=revision, token=os.environ.get("HF_TOKEN"))
        print(f"  access OK  {model}")
    except Exception as e:
        problems.append(f"{model}: {type(e).__name__}: {str(e)[:200]}")
if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Some models cannot be downloaded. For Llama: add HF_TOKEN under Add-ons -> Secrets, switch it on for "
        "this notebook, and check that https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct says you have access."
    )

## 4. Install BFCL and vLLM (about 1-5 minutes)
The same environment as notebooks 01 and 02: a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]` (vLLM 0.8.5) and the same pins.

In [ ]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch; from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

## 5. Run and score every model (about 1 hour)
One model at a time: `run_eval.py` (answers), then `score.py` (accuracy). Each run's full log is saved as `run_<run id>.log`. If a model fails, the notebook prints the end of its log and goes on with the next one.

In [ ]:
def tail(path, n=25):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


def gpu_memory_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def wait_for_free_gpus(limit_mib=1000, wait_s=120):
    """A finished run should leave the GPUs empty; wait briefly if they are not yet."""
    start = time.time()
    while max(gpu_memory_used_mib()) > limit_mib and time.time() - start < wait_s:
        time.sleep(5)
    used = gpu_memory_used_mib()
    if max(used) > limit_mib:
        print(f"  WARNING: GPU memory still in use before this run: {used} MiB")


status = {}
for config in CONFIGS:
    run_id = run_id_of(config)
    model = cfgs[config]["model"]
    log_path = f"{WORK}/run_{run_id}.log"
    wait_for_free_gpus()
    print(f"\n=== {run_id} ({model}): started {time.strftime('%H:%M:%S')} UTC", flush=True)
    start = time.time()
    try:
        with open(log_path, "w") as log:
            subprocess.run(
                [f"{VENV}/bin/python", "src/run_eval.py", "--config", config, "--mode", MODE, "--overwrite"],
                cwd=REPO, stdout=log, stderr=subprocess.STDOUT, timeout=RUN_TIMEOUT_MIN * 60,
            )
    except subprocess.TimeoutExpired:
        print(f"  stopped after {RUN_TIMEOUT_MIN} minutes")
    minutes = (time.time() - start) / 60

    if not os.path.exists(f"{REPO}/results/meta/{run_id}.json"):
        status[run_id] = "FAILED"
        print(f"  FAILED after {minutes:.1f} min. End of {log_path}:\n{tail(log_path)}")
    else:
        print(f"  answers done in {minutes:.1f} min: " +
              next((l.strip() for l in tail(log_path, 10).splitlines() if l.startswith("Done:")), ""))
        r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"results/raw/{run_id}.jsonl"],
                           cwd=REPO, capture_output=True, text=True)
        print(r.stdout[-2000:])
        status[run_id] = "OK" if r.returncode == 0 else "SCORING FAILED"
        if r.returncode != 0:
            print(r.stderr[-3000:])

    if DELETE_MODEL_AFTER_RUN:
        shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{model.replace('/', '--')}"),
                      ignore_errors=True)

print("\nSummary:")
for run_id, s in status.items():
    print(f"  {run_id:28s} {s}")
if any(s != "OK" for s in status.values()):
    print("Some runs did not finish. The results of the others are still reported and packaged below.")

## 6. Baseline table
Accuracy per category, as in BFCL. **Overall** is BFCL's Non-Live Overall Acc (mean of simple, multiple, parallel and parallel-multiple); irrelevance is reported separately. **Parsed** is the share of answers to the AST questions that were valid JSON in the requested format: answers that fail here are syntactic failures.

In [ ]:
import re

import pandas as pd

COLUMNS = {
    "simple_python": "Python", "simple_java": "Java", "simple_javascript": "JavaScript",
    "multiple": "Multiple", "parallel": "Parallel", "parallel_multiple": "Parallel Multiple",
    "non_live_overall": "Overall", "irrelevance": "Irrelevance",
}


def table_order(config):
    family = 0 if "qwen" in config else 1
    size = float(re.search(r"(\d+(?:\.\d+)?)b", os.path.basename(config)).group(1))
    return family, size


summary = pd.read_csv(f"{REPO}/results/summary.csv")
acc_rows, info_rows, metas = [], [], {}
for config in sorted(CONFIGS, key=table_order):
    run_id = run_id_of(config)
    s = summary[summary["run_id"] == run_id].set_index("category")
    if s.empty or "non_live_overall" not in s.index:
        continue
    with open(f"{REPO}/results/meta/{run_id}.json") as f:
        meta = metas[run_id] = json.load(f)
    scores = pd.read_json(f"{REPO}/results/scores/{run_id}.jsonl", lines=True)
    ast = scores[scores["category"] != "irrelevance"]
    name = meta["model"].split("/")[-1]
    acc_rows.append({"Model": name, **{label: float(s.loc[c, "accuracy"]) for c, label in COLUMNS.items()}})
    info_rows.append({
        "Model": name,
        "Parsed, AST questions (%)": round(100 * ast["parse_ok"].mean(), 2),
        "Unparsed answers": int((~ast["parse_ok"]).sum()),
        "Cut off at max_tokens": meta["finish_reasons"].get("length", 0),
        "GPUs": meta["config"]["tensor_parallel_size"],
        "Weights per GPU (GiB)": meta["memory"]["model_weights_gib"],
        "Generation (s)": meta["generation_seconds"],
        "Output tokens/s": meta["output_tokens_per_second"],
    })

accuracy = pd.DataFrame(acc_rows)
info = pd.DataFrame(info_rows)
display(accuracy)
display(info)
accuracy.to_csv(f"{WORK}/baseline_accuracy.csv", index=False)
info.to_csv(f"{WORK}/baseline_run_info.csv", index=False)

## 7. Write the report and package the results

In [ ]:
def md_table(df):
    fmt = lambda v: f"{v:.2f}" if isinstance(v, float) else str(v)
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" if i == 0 else "---:" for i in range(len(df.columns))) + "|"
    return [head, sep] + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]


first = next(iter(metas.values())) if metas else None
lines = [f"# {REPORT_TITLE}", "", f"**Date:** {date.today()}",
         "**Goal:** function-calling accuracy of every core model in full precision (FP16), the baseline for "
         "every quantized run.", ""]
if first:
    v = first["versions"]
    lines += [
        "## Setup", "",
        f"- **Questions:** BFCL `non_live`, {first['n_questions']} questions per model, bfcl-eval {v['bfcl-eval']}",
        f"- **Prompt format:** `{first['prompt_format']}` (as in Update 3), {MODE} decoding; Qwen3 with thinking off",
        f"- **Decoding:** greedy (temperature 0), seed {first['config']['sampling']['seed']}, "
        f"at most {first['config']['sampling']['max_tokens']} answer tokens",
        f"- **Inference:** vLLM {v['vllm']}, FP16, on {', '.join(first['gpus'])}; Qwen3-8B split across both T4s",
        f"- **Software:** Python {v['python']}, torch {v['torch']}, transformers {v['transformers']}",
        f"- **Code:** commit `{(first['code']['commit'] or 'unknown')[:7]}`; model revisions are in `results/meta/`",
        "- **Overall** = BFCL's Non-Live Overall Acc (unweighted mean of simple, multiple, parallel and "
        "parallel-multiple; simple = mean of Python, Java, JavaScript). Irrelevance is reported on its own.",
        "",
        "## Accuracy (%)", "",
    ] + md_table(accuracy) + ["", "## Format, length and speed", ""] + md_table(info)
failed = [r for r, s in status.items() if s != "OK"]
lines += ["", "## Notes", ""]
lines += [f"- Did not finish: {', '.join(failed)}" if failed else "- All runs finished."]
lines += ["- "]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/{REPORT_NAME}", "w") as f:
    f.write(report)
print(report)

pack = f"{WORK}/update4_results"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results", ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
for name in [REPORT_NAME, "baseline_accuracy.csv", "baseline_run_info.csv", "requirements-freeze.txt"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
for config in CONFIGS:
    log_path = f"{WORK}/run_{run_id_of(config)}.log"
    if os.path.exists(log_path):
        shutil.copy(log_path, pack)
archive = shutil.make_archive(pack, "zip", pack)
print("Saved", archive)

## After the run: what goes into the repository

1. Download **`update4_results.zip`** from the notebook's **Output** section (or the saved version's Output tab).
2. Download the notebook with its outputs (**File → Download Notebook**) and replace `notebooks/03_fp16_baselines.ipynb` with it.
3. From the zip:
   - `baseline_fp16.md` → `results/baseline_fp16.md`; write a few sentences under **Notes**
   - `results/meta/`, `results/scores/` and `results/summary.csv` → the same places in the repository
   - The raw answers in `results/raw/` stay out of git; keep the zip as a backup.
4. Add the GPU time to `notes/compute_log.md`, add the Update 4 entry at the top of `UPDATES.md`, then commit and push before the meeting.